# Q2 · 03 RoBERTa fine-tuning and pooling ablation

**Model.** Pre-trained `roberta-base` → last hidden layer → **masked mean pooling** over every non-padding token → dropout (0.1) → linear (5). The whole encoder is fine-tuned.

$$\text{pooled} = \frac{\sum_t m_t\, h_t}{\sum_t m_t}, \qquad m_t = \text{attention\_mask}_t$$

**Ablation.** The same model with **first-token pooling** (only the `<s>` vector, $h_0$). Everything else, including seeds and data order, is identical, so any gap is down to pooling.

| | |
|---|---|
| Loss | cross-entropy |
| Optimiser | AdamW, lr 2e-5, weight decay 0.01 (none on biases and LayerNorm) |
| Schedule | 6% linear warmup, then linear decay to 0 |
| Precision | fp16 autocast + GradScaler on T4 / P100, bf16 on Ampere |
| Batches | 32 reviews, grouped by length, padded per batch; gradient clipping at 1.0 |
| Epochs | 2, validation twice per epoch, best validation macro-F1 checkpoint kept |
| Seeds | 42 and 43 for each pooling (4 runs) |

The test split is not loaded in this notebook.

In [ ]:
import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")
BRANCH = "main"   # the branch to clone when this notebook runs outside the repo (e.g. imported into Kaggle)

root = Path.cwd().resolve()
while not (root / "common").is_dir() and root != root.parent:
    root = root.parent
if not (root / "common").is_dir():
    import subprocess
    root = Path.cwd() / "NLP-Group-Assignment-2"
    if not root.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Akshats-git/NLP-Group-Assignment-2.git", str(root)], check=True)
    os.chdir(root / "Question-2")
sys.path[:0] = [str(root), str(root / "Question-2")]

import torch
from common import DEFAULT_SEEDS, get_device

LIMIT = int(os.environ.get("Q2_LIMIT", 0)) or None   # set for a quick run on a random subset
DEVICE = os.environ.get("Q2_DEVICE", "")             # empty picks cuda, then mps, then cpu
print(get_device(DEVICE or None),torch.cuda.device_count(), "GPU(s)", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. The model

A forward pass on four real reviews, to check shapes and that both pooling variants are wired up. RoBERTa's padding id is 1 and the attention mask decides what the mean pooling averages over.

In [ ]:
from common import count_parameters, format_count
from q2 import roberta_data
from q2.data import load_reviews
from q2.models import RobertaClassifier

tokenizer = roberta_data.get_tokenizer()
sample = roberta_data.tokenize(load_reviews(("validation",), limit=4), tokenizer)["validation"]
batch = next(iter(roberta_data.make_loader(sample, tokenizer, batch_size=4, num_workers=0)))
print({k: tuple(v.shape) for k, v in batch.items()})

for pooling in ("mean", "first"):
    model = RobertaClassifier(pooling=pooling).eval()
    with torch.no_grad():
        logits = model(batch["input_ids"], batch["attention_mask"])
    params = count_parameters(model)
    print(f"{pooling:>5} pooling: logits {tuple(logits.shape)}, {format_count(params['trainable'])} trainable parameters")
del model
_ = gc.collect()

## 2. Training runs

Each run writes `outputs/results/<run>/` (run info, history, summary, validation predictions) and keeps its best weights in `outputs/checkpoints/<run>/best.pt`. A run that already has a `summary.json` is skipped. On Kaggle, attaching an earlier version of this notebook as an input brings its finished runs back first, so a long job can be split over several versions.

In [ ]:
from common import load_json, results_dir
from q2.kaggle import import_finished_runs
from q2.train_roberta import RobertaConfig, train

print("imported from earlier versions:", import_finished_runs("roberta_*") or "none")

runs = [RobertaConfig(pooling=pooling, seed=seed, limit=LIMIT, epochs=1 if LIMIT else 2, device=DEVICE)
        for pooling in ("mean", "first") for seed in DEFAULT_SEEDS]

summaries = {}
for config in runs:
    done = results_dir(2) / config.run_name / "summary.json"
    if done.exists():
        print(f"{config.run_name}: already trained, loading its summary")
        summaries[config.run_name] = load_json(done)
        continue
    summaries[config.run_name] = train(config)
    gc.collect()
    torch.cuda.empty_cache()

## 3. Validation results and the ablation

Mean ± standard deviation over the two seeds, at each run's best checkpoint. These are validation numbers, used for model selection. The test-set comparison comes in the evaluation step.

In [ ]:
from IPython.display import Markdown
from common import History, format_mean_std, markdown_table, save_json
from common.metrics import summarize_runs

METRICS = {"accuracy": "Accuracy (%)", "macro_f1": "Macro-F1 (%)", "mae": "MAE (stars)"}
rows, by_pooling = [], {}
for pooling, label in (("mean", "Mean pooling"), ("first", "First-token pooling")):
    vals = [summaries[c.run_name]["val"] for c in runs if c.pooling == pooling]
    summary = summarize_runs(vals)
    by_pooling[pooling] = summary
    rows.append({"RoBERTa": label,
                 **{name: format_mean_std(summary[k]["values"], scale=1 if k == "mae" else 100, digits=2 if k != "mae" else 3)
                    for k, name in METRICS.items()},
                 "best epoch": ", ".join(str(summaries[c.run_name]["best_epoch"]) for c in runs if c.pooling == pooling)})
if LIMIT is None:
    save_json(by_pooling, results_dir(2) / "roberta_validation_summary.json")
Markdown(markdown_table(rows))

In [ ]:
from common import figures_dir
from common.plotting import plot_history

seed = DEFAULT_SEEDS[0]
histories = {label: History.load(results_dir(2) / RobertaConfig(pooling=p, seed=seed, limit=LIMIT).run_name / "history.json")
             for p, label in (("mean", "Mean pooling"), ("first", "First-token"))}
out = figures_dir(2) / "roberta_ablation_curves.png" if LIMIT is None else None
plot_history(histories, metrics=("loss", "macro_f1"), out_path=out,
             title=f"RoBERTa: mean vs first-token pooling (seed {seed})")

Train curves (dashed) are averaged over the batches since the previous evaluation, with dropout on, so they are a rough signal. Validation curves (solid) are measured on the full 5,000-review validation split.